# Replicando análisis de FoodProK en Stata


## Notas de MARKDOWN 

# Título grande
## Título mediano
### Título pequeño

Texto normal en párrafo

**texto en negritas**

*texto en cursiva*

- punto de lista
- otro punto

In [1]:
# Importar librerías cada vez que quiera usar Python
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Pedirle print para verificar que todo se instaló correctamente
print("Todo instalado correctamente ✓")
print(f"Pandas versión: {pd.__version__}")

Todo instalado correctamente ✓
Pandas versión: 3.0.1


# REVISAR ARCHIVOS E IMPORTAR BASES DE DATOS

In [2]:
# Verificar Nombre de archivos antes de importar las bases
import os


# Mi notebook está guardado en la carpeta notebooks
## Iniciar el path con ../ para indicarle a Python que "suba" un nivel en los folders
archivos = os.listdir("../data/raw/")
print(archivos)

['IFPS2020_allcountry_analytic_20230718.dta', 'IFPS2021_allcountry_analytic_20230717.dta', 'IFPS2022_allcountry_analytic_20230830.dta']


In [3]:
# Cargar bases de datos como data frames

# Cargar 2020
df_2020 = pd.read_stata("../data/raw/IFPS2020_allcountry_analytic_20230718.dta", convert_categoricals=False)

    ## Stata permite etiquetas de valor duplicadas, python NO.
    ## convert_categoricals=False evita este error

# Cargar 2021
df_2021 = pd.read_stata(
    "../data/raw/IFPS2021_allcountry_analytic_20230717.dta",
    convert_categoricals=False
)

# Cargar 2022
df_2022 = pd.read_stata(
    "../data/raw/IFPS2022_allcountry_analytic_20230830.dta",
    convert_categoricals=False
)

In [4]:
# Ver y print cantidad de filas y columnas antes de unir
print(f"2020: {df_2020.shape}")
print(f"2021: {df_2021.shape}")
print(f"2022: {df_2022.shape}")

2020: (21753, 1833)
2021: (26285, 1828)
2022: (26273, 1948)


Nota: Cada base tiene diferente número de observaciones (porque no es longitudinal) y diferente número de variables (se agregan o eliminan preguntas cada año)

In [5]:
# Ver solo los nombres de todas las columnas en 2020 (como referencia)
df_2020.columns.tolist()

['ID',
 'wght',
 'W3W4_match_DV',
 'W3_ID_match_DV',
 'W2W3W4_match_DV',
 'W2_ID_match_DV',
 'W1W2W3W4_match_DV',
 'W1_ID_match_DV',
 'country',
 'sample_USA',
 'language',
 'Vdatesub',
 'TimeStarted',
 'DateSubmitted',
 'monthsub_DV',
 'status_DV',
 'respstatus_DV',
 'expletive_DV',
 'mobilebrowser',
 'age',
 'age_DV',
 'sex',
 'consent',
 'gender',
 'gender_otext',
 'gender_DV',
 'student',
 'occup',
 'occup_otext',
 'occup_DV',
 'occup_covid_DV',
 'child_any',
 'child_home',
 'child_home_DV',
 'child_home_U18_DV',
 'child_hhld_DV',
 'child_DQ_DV',
 'child1_age',
 'child1_age_DKR',
 'child2_age',
 'child2_age_DKR',
 'child3_age',
 'child3_age_DKR',
 'child4_age',
 'child4_age_DKR',
 'child5_age',
 'child5_age_DKR',
 'child6_age',
 'child6_age_DKR',
 'child7_age',
 'child7_age_DKR',
 'child8_age',
 'child8_age_DKR',
 'child9_age',
 'child9_age_DKR',
 'child10_age',
 'child10_age_DKR',
 'child1_age_DV',
 'child2_age_DV',
 'child3_age_DV',
 'child4_age_DV',
 'child5_age_DV',
 'child6_ag

In [6]:
# Explorar AÑO
print('year' in df_2020.columns)
print('year' in df_2021.columns)
print('year' in df_2022.columns)

False
False
False


Nota: Al explorar las bases, confirmé que a variable "year" arrojó FALSE en los 3 años.

Esto significa que la variable AÑO no existe en las bases.

La creo manualmente antes de unir para poder identificar cada observación por año después.

In [7]:
# Crear variable year en cada base antes de unir
df_2020['year'] = 2020
df_2021['year'] = 2021
df_2022['year'] = 2022

C:\Users\kathi\AppData\Local\Temp\ipykernel_25168\3571457639.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_2020['year'] = 2020
C:\Users\kathi\AppData\Local\Temp\ipykernel_25168\3571457639.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_2021['year'] = 2021
C:\Users\kathi\AppData\Local\Temp\ipykernel_25168\3571457639.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.co

In [8]:
# Unir bases de datos (equivalente al append de Stata)

df = pd.concat([df_2020, df_2021, df_2022], ignore_index=True)
    # ignore_index=True reinicia la numeración de filas desde 0

# Verificar que quedó bien
print(df.shape)
print(df['year'].value_counts())

(74311, 2210)
year
2021    26285
2022    26273
2020    21753
Name: count, dtype: int64


La base unificada tiene 74,311 participantes y 2,210 columnas

2021: **26,285** → participantes de 2021

2022: **26,273** → participantes de 2022

2020: **21,753** → participantes de 2020

Suma: **21,753** + 26,285 + 26,273 = **74,311**

# Explorar variables de interés
## País

In [9]:
# Explorar variable Country
print(df['country'].value_counts())
print(df['country'].value_counts(normalize=True).round(3) * 100)

country
4.0    19390
5.0    16355
1.0    13318
3.0    12648
2.0    12600
Name: count, dtype: int64
country
4.0    26.1
5.0    22.0
1.0    17.9
3.0    17.0
2.0    17.0
Name: proportion, dtype: float64


Nota:

1 Canada 13318 obs

2 Australia 12600 obs

3 UK 12648 obs

4 USA 19390 obs

5 Mexico 16355 obs

Total: 74311 obs

In [10]:
# Convertir códigos numéricos a nombres de países
df['country'] = df['country'].astype("Int64")

country_labels = {1: 'Canada', 2: 'Australia', 3: 'UK', 4: 'USA', 5: 'Mexico'}
df['country'] = df['country'].map(country_labels)
    # .map() Es el equivalente del recode + label define de Stata —> toma cada valor numérico y lo reemplaza con su etiqueta.

# Definir orden alfabético con Australia como referencia
df['country'] = pd.Categorical(
    df['country'],
    categories=['Australia', 'Canada', 'Mexico', 'UK', 'USA'],
    ordered=False
)

# Verificar — debe coincidir con do file
print(df['country'].cat.categories)   # Muestra el orden asignado
print(df['country'].value_counts().sort_index())  # Imprimir valores con el orden del índice - Los valores deben coincidir con el do file

Index(['Australia', 'Canada', 'Mexico', 'UK', 'USA'], dtype='str')
country
Australia    12600
Canada       13318
Mexico       16355
UK           12648
USA          19390
Name: count, dtype: int64


## Edad

In [11]:
# Explorar variable age
print(df['age'].describe())  # .describe da media, desviación estándar, mínimo, máximo y percentile
print(f"\nMissings: {df['age'].isnull().sum()}") ## isnull para identificar los missing

count    74311.000000
mean        45.361010
std         16.850802
min         18.000000
25%         31.000000
50%         45.000000
75%         59.000000
max         99.000000
Name: age, dtype: float64

Missings: 0


Note: No hay missings en Edad

## Sexo

In [12]:
# Explorar variable sex
print(df['sex'].value_counts())
print(f"\nMissings: {df['sex'].isnull().sum()}")

sex
2.0    38043
1.0    36268
Name: count, dtype: int64

Missings: 0


Nota: No hay missings en sexo

In [13]:
# Mapear sexo a texto
sex_labels = {1: 'Male', 2: 'Female'}
df['sex'] = df['sex'].map(sex_labels)

# Verificar
print(df['sex'].value_counts())

sex
Female    38043
Male      36268
Name: count, dtype: int64


## Educación

In [14]:
# Explorar variable education
print(df['educ_DV'].value_counts())
print(f"\nMissings: {df['educ_DV'].isnull().sum()}")

educ_DV
 3.0     28592
 1.0     26411
 2.0     18986
-99.0      322
Name: count, dtype: int64

Missings: 0


In [15]:
# Guardar tamaño antes
n_antes = len(df)

# Eliminar "Not stated" (-99)
df = df[df['educ_DV'] != -99]

# Verificar cuántos se eliminaron
# Mostrar cuántos se eliminaron automáticamente
print(f"Eliminados: {n_antes - len(df)}")
print(f"Observaciones restantes: {len(df)}")

# Mapear a texto con orden lógico
educ_labels = {1: 'Low', 2: 'Medium', 3: 'High'}
df['educ_DV'] = df['educ_DV'].map(educ_labels)

# Definir orden lógico (no alfabético — aquí el orden importa)
df['educ_DV'] = pd.Categorical(
    df['educ_DV'],
    categories=['Low', 'Medium', 'High'],
    ordered=True
)

print(df['educ_DV'].value_counts().sort_index())

Eliminados: 322
Observaciones restantes: 73989
educ_DV
Low       26411
Medium    18986
High      28592
Name: count, dtype: int64


## Etnicidad

In [16]:
# Explorar etnicidad
print(df['eth_DV'].value_counts())
print(f"\nMissings: {df['eth_DV'].isnull().sum()}")

eth_DV
 1.0     53275
 2.0     19989
-99.0      725
Name: count, dtype: int64

Missings: 0


In [17]:
# Guardar tamaño antes
n_antes = len(df)

# Eliminar "Not stated" (-99)
df = df[df['eth_DV'] != -99]

# Verificar cuántos se eliminaron
# Mostrar cuántos se eliminaron automáticamente
print(f"Eliminados: {n_antes - len(df)}")
print(f"Observaciones restantes: {len(df)}")

# Mapear a texto con orden lógico
eth_labels = {1: 'Majority', 2: 'Minority'}
df['eth_DV'] = df['eth_DV'].map(eth_labels)

# Definir orden lógico 
df['eth_DV'] = pd.Categorical(
    df['eth_DV'],
    categories=['Majority', 'Minority'],
    ordered=False
)

print(df['eth_DV'].value_counts().sort_index())

Eliminados: 725
Observaciones restantes: 73264
eth_DV
Majority    53275
Minority    19989
Name: count, dtype: int64


## Income Adequacy

In [18]:
# Explorar Income adequacy
print(df['income_adeq'].value_counts())
print(f"\nMissings: {df['income_adeq'].isnull().sum()}")

income_adeq
 3.0     26656
 2.0     15630
 4.0     15410
 5.0      8597
 1.0      6293
-77.0      415
-88.0      263
Name: count, dtype: int64

Missings: 0


In [19]:
# Guardar tamaño antes
n_antes = len(df)

# Eliminar DK y RTA (-77, -88)
df = df[df['income_adeq'] != -88]
df = df[df['income_adeq'] != -77]

# Verificar cuántos se eliminaron
# Mostrar cuántos se eliminaron automáticamente
print(f"Eliminados: {n_antes - len(df)}")
print(f"Observaciones restantes: {len(df)}")

# Recodificar 5 categorías a 3
income_labels = {
    1: 'Difficult',
    2: 'Difficult',
    3: 'Neither',
    4: 'Easy',
    5: 'Easy'
}
df['income_adeq'] = df['income_adeq'].map(income_labels)

# Definir orden lógico
df['income_adeq'] = pd.Categorical(
    df['income_adeq'],
    categories=['Difficult', 'Neither', 'Easy'],
    ordered=True
)

# Verificar
print(df['income_adeq'].value_counts().sort_index())

Eliminados: 678
Observaciones restantes: 72586
income_adeq
Difficult    21923
Neither      26656
Easy         24007
Name: count, dtype: int64


# Limpiar y Construir FoodProK